# Практичне заняття 4.15. Варіант 1. Пошук службових повідомлень за змістом
**Виконав:** Щербаков Богдан, навчальна група 5202.

**Мета:** побудувати невеликий пошуковий модуль з TF–IDF і косинусною схожістю, перевірити релевантність результатів і визначити обмеження методу.

**Порядок роботи:** очищення тексту → векторизація TF–IDF → пошук схожих документів → перевірка релевантності → власне поліпшення.

**Дані:** `Варіант_1_повідомлення_30.csv` — 30 синтетичних службових повідомлень (5 тем по 6 повідомлень). Відомостей обмеженого доступу немає.

In [1]:
from pathlib import Path
import csv
from collections import Counter

DATA = Path("Варіант_1_повідомлення_30.csv")
if not DATA.exists():
    DATA = Path("../Варіант_1_повідомлення_30.csv")
with DATA.open(encoding="utf-8-sig", newline="") as f:
    docs = list(csv.DictReader(f))
print("Завантажено повідомлень:", len(docs))
print("Теми:", dict(Counter(d["тема_для_перевірки"] for d in docs)))
for row in docs[:5]:
    print(row["id"], row["текст"])

Завантажено повідомлень: 30
Теми: {'Забезпечення': 6, 'Технічна підтримка': 6, 'Звітність': 6, 'Навчання': 6, 'Ремонт': 6}
M001 До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M002 Підрозділ подав заявку на десять мережевих кабелів для проведення занять.
M003 Отримано комплект витратних матеріалів та подовжувачів для навчальної аудиторії.
M004 Просимо забезпечити клас новими гарнітурами для дистанційного навчання.
M005 Уточнено потребу у папері та картриджах для друку навчальних матеріалів.


## 1. Перетворюємо тексти на вектори
`fit_transform()` будує словник за 30 повідомленнями і матрицю TF–IDF. Запит лише перетворюється (`transform`) за тим самим словником, окремий `fit_transform()` для нього не потрібен.

`ngram_range=(1, 2)` — у словник потрапляють окремі слова та пари сусідніх слів.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
X = vectorizer.fit_transform([d["текст"] for d in docs])
print("Розмір TF–IDF матриці:", X.shape, "(документи × ознаки)")

Розмір TF–IDF матриці: (30, 385) (документи × ознаки)


## 2. Пошукова функція
Косинусна схожість показує, наскільки збігаються слова запиту і документа. **Це не ймовірність правдивості** повідомлення і не гарантія, що документ відповідає потребі.

До функції викладача додано: тему з CSV (лише для перевірки), повернення результату для подальшого аналізу та позначку, якщо схожість дорівнює 0. Такий документ потрапив у топ лише тому, що треба було показати три рядки.

In [3]:
def search(query, top_k=3, vec=None, matrix=None, show=True):
    vec = vec or vectorizer
    matrix = X if matrix is None else matrix
    scores = cosine_similarity(vec.transform([query]), matrix)[0]
    selected = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)[:top_k]
    if show:
        for i in selected:
            mark = "   ← схожість 0: спільних слів немає" if scores[i] == 0 else ""
            print(f'{docs[i]["id"]}: {scores[i]:.3f} [{docs[i]["тема_для_перевірки"]}] — {docs[i]["текст"]}{mark}')
    return [docs[i]["id"] for i in selected]

query = "постачання кабелів зв’язку"
search(query, top_k=3);

M001: 0.568 [Забезпечення] — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M002: 0.073 [Забезпечення] — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.
M003: 0.000 [Забезпечення] — Отримано комплект витратних матеріалів та подовжувачів для навчальної аудиторії.   ← схожість 0: спільних слів немає


## 3. Обов'язкові експерименти
Три запити із завдання, для кожного — топ-3 ID.

In [4]:
QUERIES = ["постачання кабелів зв’язку", "помилка авторизації", "аналітичний звіт про навчання"]
top3_words = {}
for q in QUERIES:
    print("\nЗапит:", q)
    top3_words[q] = search(q)


Запит: постачання кабелів зв’язку
M001: 0.568 [Забезпечення] — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M002: 0.073 [Забезпечення] — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.
M003: 0.000 [Забезпечення] — Отримано комплект витратних матеріалів та подовжувачів для навчальної аудиторії.   ← схожість 0: спільних слів немає

Запит: помилка авторизації
M007: 0.420 [Технічна підтримка] — Під час входу до навчальної платформи виникла помилка авторизації користувача.
M001: 0.000 [Забезпечення] — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.   ← схожість 0: спільних слів немає
M002: 0.000 [Забезпечення] — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.   ← схожість 0: спільних слів немає

Запит: аналітичний звіт про навчання
M013: 0.558 [Звітність] — Підготовлено аналітичний звіт про результати навчального тижня.
M014: 0.247 [Звітність] — До кафедри подано міс

**Додатковий експеримент:** `помилка авторизації` проти `несправність входу` — однакова потреба (користувач не може увійти), різні слова.

In [5]:
for q in ["помилка авторизації", "несправність входу"]:
    print("\nЗапит:", q)
    top3_words[q] = search(q)

analyzer = vectorizer.build_analyzer()
for q in ["помилка авторизації", "несправність входу"]:
    print(f"\nОзнаки запиту «{q}», які є в словнику:", [t for t in analyzer(q) if t in vectorizer.vocabulary_])


Запит: помилка авторизації
M007: 0.420 [Технічна підтримка] — Під час входу до навчальної платформи виникла помилка авторизації користувача.
M001: 0.000 [Забезпечення] — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.   ← схожість 0: спільних слів немає
M002: 0.000 [Забезпечення] — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.   ← схожість 0: спільних слів немає

Запит: несправність входу
M012: 0.490 [Технічна підтримка] — Виникла несправність входу до електронного журналу занять.
M007: 0.115 [Технічна підтримка] — Під час входу до навчальної платформи виникла помилка авторизації користувача.
M001: 0.000 [Забезпечення] — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.   ← схожість 0: спільних слів немає

Ознаки запиту «помилка авторизації», які є в словнику: ['помилка', 'авторизації', 'помилка авторизації']

Ознаки запиту «несправність входу», які є в словнику: ['несправність', 'входу', '

### Перевірка релевантності вручну
Нижче — повідомлення, які **за змістом** відповідають кожній потребі. Я відібрав їх, прочитавши всі 30 текстів, а не за колонкою теми.

| Запит | Релевантні повідомлення | Чому |
|---|---|---|
| постачання кабелів зв’язку | M001, M002 | потреба в кабелях зв'язку / мережевих кабелях. M030 («замінити кабель живлення») — ремонт, інша потреба |
| помилка авторизації | M007, M008, M012 | у всіх трьох користувач не може увійти до системи |
| аналітичний звіт про навчання | M013, M014, M016, M018 | звіти й довідки про навчальний процес |
| несправність входу | M007, M008, M012 | та сама потреба, що й у запиті про авторизацію |

Метрика — **точність у топ-3 (precision@3)**: частка релевантних серед трьох показаних.

In [6]:
RELEVANT = {
    "постачання кабелів зв’язку": {"M001", "M002"},
    "помилка авторизації": {"M007", "M008", "M012"},
    "аналітичний звіт про навчання": {"M013", "M014", "M016", "M018"},
    "несправність входу": {"M007", "M008", "M012"},
}

def report(top3, title):
    print(title)
    total = 0
    for q, rel in RELEVANT.items():
        hits = [i for i in top3[q] if i in rel]
        missed = sorted(rel - set(top3[q]))
        total += len(hits) / 3
        print(f"  {q:32} топ-3 {top3[q]}  влучних {len(hits)}/3  пропущено {missed}")
    print(f"  Середня precision@3: {total / len(RELEVANT):.2f}")

report(top3_words, "TF–IDF за словами (базовий варіант):")

TF–IDF за словами (базовий варіант):
  постачання кабелів зв’язку       топ-3 ['M001', 'M002', 'M003']  влучних 2/3  пропущено []
  помилка авторизації              топ-3 ['M007', 'M001', 'M002']  влучних 1/3  пропущено ['M008', 'M012']
  аналітичний звіт про навчання    топ-3 ['M013', 'M014', 'M004']  влучних 2/3  пропущено ['M016', 'M018']
  несправність входу               топ-3 ['M012', 'M007', 'M001']  влучних 2/3  пропущено ['M008']
  Середня precision@3: 0.58


**Що видно:**
- **Кабелі:** M001 і M002 знайдено, але третій рядок (M003) має схожість 0 і потрапив у топ випадково. Слово «кабелів» з M002 збіглося, а «кабель» у M030 — ні: для TF–IDF різні відмінки — різні слова.
- **Авторизація:** знайдено лише M007, бо тільки в ньому є слова «помилка авторизації». M008 («не може увійти») і M012 («несправність входу») пропущено, хоча вони саме про це. Решта топу має схожість 0.
- **Звіт:** M013 і M014 релевантні, а M004 (гарнітури для «дистанційного навчання») — ні, він потрапив через спільне слово «навчання». M016 («Аналітична довідка») пропущено: «аналітична» ≠ «аналітичний».
- **Синоніми:** запит «несправність входу» знаходить M012 і M007 (спільне слово «входу»), але не M008. Запити з однаковим змістом дають різні результати. **TF–IDF не знаходить синонімів без спільних слів.**
- Апостроф у «зв’язку» розбиває слово на «зв» і «язку». Це видно в ознаках запиту.

In [7]:
print("Ознаки запиту «постачання кабелів зв’язку»:", analyzer("постачання кабелів зв’язку"))
print("Чи є в словнику «кабель» / «кабелів» / «кабелі»:",
      [w in vectorizer.vocabulary_ for w in ["кабель", "кабелів", "кабелі"]])

Ознаки запиту «постачання кабелів зв’язку»: ['постачання', 'кабелів', 'зв', 'язку', 'постачання кабелів', 'кабелів зв', 'зв язку']
Чи є в словнику «кабель» / «кабелів» / «кабелі»: [True, True, False]


## 4. Власне поліпшення: символьні n-грами
Українська мова має багато словоформ («кабель / кабелів / кабелі», «аналітичний / аналітична»). Замість цілих слів можна порівнювати **фрагменти по 3–5 символів усередині слів** (`analyzer="char_wb"`). Тоді «кабелів» і «кабель» мають спільні фрагменти «кабе», «кабел».

In [8]:
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
X_char = char_vec.fit_transform([d["текст"] for d in docs])
print("Розмір матриці символьних n-грам:", X_char.shape)

top3_char = {}
for q in RELEVANT:
    print("\nЗапит:", q)
    top3_char[q] = search(q, vec=char_vec, matrix=X_char)

Розмір матриці символьних n-грам: (30, 2826)

Запит: постачання кабелів зв’язку
M001: 0.617 [Забезпечення] — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M002: 0.172 [Забезпечення] — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.
M030: 0.101 [Ремонт] — Для відновлення роботи обладнання необхідно замінити кабель живлення.

Запит: помилка авторизації
M007: 0.563 [Технічна підтримка] — Під час входу до навчальної платформи виникла помилка авторизації користувача.
M021: 0.194 [Навчання] — Проведено демонстрацію векторизації текстових документів методом TF–IDF.
M020: 0.073 [Навчання] — На практичному занятті вивчено метод токенізації українських повідомлень.

Запит: аналітичний звіт про навчання
M013: 0.548 [Звітність] — Підготовлено аналітичний звіт про результати навчального тижня.
M016: 0.259 [Звітність] — Аналітична довідка містить висновки про відвідування та успішність слухачів.
M014: 0.248 [Звітність] — До кафедри подан

In [9]:
report(top3_words, "TF–IDF за словами:")
print()
report(top3_char, "TF–IDF за символьними n-грамами:")

TF–IDF за словами:
  постачання кабелів зв’язку       топ-3 ['M001', 'M002', 'M003']  влучних 2/3  пропущено []
  помилка авторизації              топ-3 ['M007', 'M001', 'M002']  влучних 1/3  пропущено ['M008', 'M012']
  аналітичний звіт про навчання    топ-3 ['M013', 'M014', 'M004']  влучних 2/3  пропущено ['M016', 'M018']
  несправність входу               топ-3 ['M012', 'M007', 'M001']  влучних 2/3  пропущено ['M008']
  Середня precision@3: 0.58

TF–IDF за символьними n-грамами:
  постачання кабелів зв’язку       топ-3 ['M001', 'M002', 'M030']  влучних 2/3  пропущено []
  помилка авторизації              топ-3 ['M007', 'M021', 'M020']  влучних 1/3  пропущено ['M008', 'M012']
  аналітичний звіт про навчання    топ-3 ['M013', 'M016', 'M014']  влучних 3/3  пропущено ['M018']
  несправність входу               топ-3 ['M012', 'M027', 'M007']  влучних 2/3  пропущено ['M008']
  Середня precision@3: 0.67


**Результат поліпшення.** Символьні n-грами краще впоралися зі словоформами: для «аналітичного звіту» знайдено M016 і всі три результати релевантні; запит «кабелі», якого словниковий TF–IDF взагалі не розумів, тепер знаходить M001 і M002 (перевірка нижче).

Але з'явилися **нові хибні збіги**: на «помилку авторизації» другим іде M021 («векторизації») через спільне закінчення «-ації», а на «несправність входу» — M027 («несправного ноутбука»). M008 («не може увійти») не знаходиться жодним із двох способів: спільних фрагментів зі словами запиту немає. **Синонімії символьні n-грами не вирішують.**

In [10]:
for q in ["кабелі", "не вдається увійти"]:
    print(f"\nЗапит «{q}» — за словами:")
    search(q, vec=vectorizer, matrix=X)
    print(f"Запит «{q}» — символьні n-грами:")
    search(q, vec=char_vec, matrix=X_char)


Запит «кабелі» — за словами:
M001: 0.000 [Забезпечення] — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.   ← схожість 0: спільних слів немає
M002: 0.000 [Забезпечення] — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.   ← схожість 0: спільних слів немає
M003: 0.000 [Забезпечення] — Отримано комплект витратних матеріалів та подовжувачів для навчальної аудиторії.   ← схожість 0: спільних слів немає
Запит «кабелі» — символьні n-грами:
M002: 0.238 [Забезпечення] — Підрозділ подав заявку на десять мережевих кабелів для проведення занять.
M001: 0.229 [Забезпечення] — До навчального центру потрібне постачання кабелів зв’язку для комп’ютерного класу.
M030: 0.161 [Ремонт] — Для відновлення роботи обладнання необхідно замінити кабель живлення.

Запит «не вдається увійти» — за словами:
M008: 0.303 [Технічна підтримка] — Слухач не може увійти до особистого кабінету через невірний пароль.
M009: 0.138 [Технічна підтримка] — Система диста

## 5. Відповіді на запитання

**Чи всі три відповіді справді відповідають інформаційній потребі?** Ні. Для кабелів релевантні 2 з 3 (третій має схожість 0), для авторизації — 1 з 3, для аналітичного звіту — 2 з 3 (M004 про гарнітури потрапив через слово «навчання»). Метод завжди повертає рівно три рядки, навіть якщо схожих документів менше, тому кожен результат треба перевіряти.

**Чи знаходить TF–IDF синонімічні формулювання без спільних слів?** Ні. «Помилка авторизації», «несправність входу» і «не може увійти» описують одну проблему, але кожен запит знаходить лише документ зі своїми словами. Навіть різні відмінки одного слова («кабель» / «кабелів») для словникового TF–IDF — різні ознаки.

**Чому назва теми в CSV не повинна замінювати перевірку змісту?** Тема — це загальна рубрика, а не відповідь на конкретну потребу. M030 має тему «Ремонт» і містить «кабель», але це заміна кабелю живлення, а не постачання кабелів зв'язку. M004 має тему «Забезпечення», але стосується гарнітур. В одній темі є повідомлення з різними потребами, а одна потреба (вхід у систему) може бути описана різними словами. Крім того, в реальних повідомленнях теми може не бути або її вказано помилково.

## 6. Звіт слухача

**Запит, який я перевіряв:** `помилка авторизації` (і порівняння з `несправність входу`), а також усі три обов'язкові запити.

**Топ-3 ID документів (TF–IDF за словами):**
- `постачання кабелів зв’язку` → M001 (0,568), M002 (0,073), M003 (0,000)
- `помилка авторизації` → M007 (0,420), далі документи зі схожістю 0
- `аналітичний звіт про навчання` → M013 (0,558), M014 (0,247), M004 (0,096)
- `несправність входу` → M012 (0,490), M007 (0,115), далі схожість 0

**Відповідні та невідповідні документи:** відповідні — M001, M002, M007, M012, M013, M014. Невідповідні — M003 і решта документів зі схожістю 0, а також M004 (збіг лише за словом «навчання»). Пропущені релевантні — M008, M012 (для «помилки авторизації»), M016, M018. Середня precision@3 базового методу за чотирма запитами — 0,58.

**Одне обмеження TF–IDF:** метод порівнює лише однакові слова. Він не знаходить синонімів («несправність входу» ≠ «помилка авторизації»), не розпізнає інші словоформи («кабель» ≠ «кабелів») і завжди повертає top_k результатів, навіть зі схожістю 0. Висока схожість означає збіг слів, а не правильність чи релевантність повідомлення.

**Моя пропозиція поліпшення:**
1. **Символьні n-грами** (`analyzer="char_wb"`, 3–5 символів) — перевірено вище: precision@3 зросла з 0,58 до 0,67, знайдено M016, запит «кабелі» почав працювати. Ціна — нові хибні збіги за спільними частинами слів (M021 «векторизації», M027 «несправного»).
2. **Поріг схожості:** не показувати результати нижче ≈ 0,1 і явно писати «схожих повідомлень не знайдено».
3. **Синоніми:** словник службових синонімів (вхід / авторизація / увійти) або лематизація. Ще краще — семантичні векторні представлення (наприклад, багатомовні моделі sentence-transformers), які порівнюють зміст, а не літери.
4. Результат пошуку — лише підказка: остаточну релевантність перевіряє людина.

**Висновок.** Пошук на TF–IDF простий, швидкий і прозорий: для кожного результату видно, які слова збіглися. На 30 повідомленнях він надійно знаходить документи з тими самими словами, що й у запиті. Але він не розуміє змісту: пропускає синоніми й інші словоформи та підтягує документи за випадковими спільними словами. Тому такий модуль підходить як перший фільтр, а відібрані повідомлення має перевіряти оператор.